In [9]:
import os
downloads = os.path.expanduser("~/Downloads")
print([f for f in os.listdir(downloads) if f.lower().endswith(".csv")])

['Urbana_Open Expenditures_ Payment Details.csv', 'Pinto_199_PC3.csv', 'p34w-i7rn.csv', '2025-12-21T0006_Grades-math_199_120258_253722.csv', 'binary.csv', '2025-12-19T1956_Grades-math_199_120258_253722.csv', 'p34w-i7rn (1).csv', 'City Budget and Spending.csv', 'binary 2.csv']


In [11]:
import os, pandas as pd

downloads = os.path.expanduser("~/Downloads")

budget_path = os.path.join(downloads, "City Budget and Spending.csv")

# pick the first CSV that contains both "Urbana" and "Payment Details"
payments_name = [f for f in os.listdir(downloads) if ("Urbana" in f and "Payment Details" in f and f.endswith(".csv"))][0]
payments_path = os.path.join(downloads, payments_name)

print("Using payments file:", payments_name)

budget = pd.read_csv(budget_path)
payments = pd.read_csv(payments_path)

print("BUDGET SHAPE:", budget.shape)
print("PAYMENTS SHAPE:", payments.shape)

print("\nBUDGET columns:", list(budget.columns))
print("\nPAYMENTS columns:", list(payments.columns))



Using payments file: Urbana_Open Expenditures_ Payment Details.csv
BUDGET SHAPE: (1000, 22)
PAYMENTS SHAPE: (1000, 25)

BUDGET columns: ['full_account', 'account_description', 'budget_year', 'account_type', 'organization_code', 'object_code', 'object_description', 'project_code', 'project_description', 'fund_code', 'fund_description', 'function_code', 'function_description', 'department_code', 'department_description', 'program_code', 'program_description', 'original_budget', 'revised_budget', 'actual_balance', 'budget_remaining', 'percent_remaining']

PAYMENTS columns: ['fiscal_year', 'fiscal_year_period', 'department', 'division', 'program', 'budget_category', 'expense_category', 'fund', 'fund_type', 'vendor', 'vendor_id', 'vendor_zip', 'payment_id', 'payment_method', 'payment_date', 'payment_status', 'invoice_id', 'invoice_line_number', 'invoice_line_distribution', 'invoice_date', 'amount', 'description', 'contract_number', 'purchase_order_number', 'purchase_order_line_number']


In [12]:
import re
import pandas as pd

def norm_dept(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

budget = budget.copy()
payments = payments.copy()

budget["dept_norm"] = budget["department_description"].map(norm_dept)
payments["dept_norm"] = payments["department"].map(norm_dept)

print("Example cleaned budget departments:")
print(budget["dept_norm"].dropna().unique()[:10])

print("\nExample cleaned payment departments:")
print(payments["dept_norm"].dropna().unique()[:10])

Example cleaned budget departments:
['EXECUTIVE' 'PUBLIC WORKS' 'COMMUNITY DEVELOPMENT' '' 'CITY COUNCIL'
 'CITY CLERK' 'HUMAN RESOURCES FINANCE DEPT']

Example cleaned payment departments:
['NOT APPLICABLE' 'GENERAL SERVICES' 'POLICE' 'COMMUNITY DEVELOPMENT'
 'PUBLIC WORKS' 'FIRE' 'CITY CLERK' 'EXECUTIVE'
 'HUMAN RESOURCES FINANCE DEPT']


In [13]:
!pip -q install thefuzz python-Levenshtein


In [14]:
from thefuzz import process, fuzz

budget_depts = budget["dept_norm"].dropna().unique().tolist()

def match_dept(s, choices, threshold=85):
    if not s:
        return ("", 0)
    best, score = process.extractOne(s, choices, scorer=fuzz.token_set_ratio)
    return (best, score) if score >= threshold else ("", score)

matches = payments["dept_norm"].apply(lambda s: match_dept(s, budget_depts))
payments["budget_dept_norm"] = matches.apply(lambda t: t[0])
payments["dept_match_score"] = matches.apply(lambda t: t[1])

print("Matched percentage:", (payments["budget_dept_norm"] != "").mean())

payments[["department","dept_norm","budget_dept_norm","dept_match_score"]].head(20)


Matched percentage: 0.706


,department,dept_norm,budget_dept_norm,dept_match_score
0,NOT APPLICABLE,NOT APPLICABLE,,46
1,GENERAL SERVICES,GENERAL SERVICES,,45
2,NOT APPLICABLE,NOT APPLICABLE,,46
3,POLICE,POLICE,,44
4,COMMUNITY DEVELOPMENT,COMMUNITY DEVELOPMENT,COMMUNITY DEVELOPMENT,100
5,POLICE,POLICE,,44
6,PUBLIC WORKS,PUBLIC WORKS,PUBLIC WORKS,100
7,PUBLIC WORKS,PUBLIC WORKS,PUBLIC WORKS,100
8,POLICE,POLICE,,44
9,PUBLIC WORKS,PUBLIC WORKS,PUBLIC WORKS,100


In [16]:
# Every payment row
#+ its department’s budget info


In [17]:
budget_agg = (budget
    .groupby("dept_norm", as_index=False)
    .agg(
        revised_budget=("revised_budget","sum"),
        original_budget=("original_budget","sum"),
        actual_balance=("actual_balance","sum"),
        budget_remaining=("budget_remaining","sum"),
        percent_remaining=("percent_remaining","mean")
    )
)

budget_agg.head()


,dept_norm,revised_budget,original_budget,actual_balance,budget_remaining,percent_remaining
0,,-144408906.8,-140107738.8,-114794077.0,-30521337.6,3473.359617
1,CITY CLERK,628409.0,626024.0,431031.7,128435.8,1094.506508
2,CITY COUNCIL,184254.0,183279.0,136165.8,33482.9,695.645714
3,COMMUNITY DEVELOPMENT,2026031.0,1587821.0,939441.9,625993.9,51177.414444
4,EXECUTIVE,6887759.1,6547573.8,4980290.4,1286783.8,1450.315397


In [18]:
master = payments.merge(
    budget_agg,
    left_on="budget_dept_norm",
    right_on="dept_norm",
    how="left"
)

print("MASTER SHAPE:", master.shape)
print("Missing budget info %:", master["revised_budget"].isna().mean())

master[[
    "department",
    "budget_dept_norm",
    "dept_match_score",
    "revised_budget",
    "percent_remaining"
]].head(15)


MASTER SHAPE: (1000, 34)
Missing budget info %: 0.0


,department,budget_dept_norm,dept_match_score,revised_budget,percent_remaining
0,NOT APPLICABLE,,46,-144408906.8,3473.359617
1,GENERAL SERVICES,,45,-144408906.8,3473.359617
2,NOT APPLICABLE,,46,-144408906.8,3473.359617
3,POLICE,,44,-144408906.8,3473.359617
4,COMMUNITY DEVELOPMENT,COMMUNITY DEVELOPMENT,100,2026031.0,51177.414444
5,POLICE,,44,-144408906.8,3473.359617
6,PUBLIC WORKS,PUBLIC WORKS,100,0.0,0.000000
7,PUBLIC WORKS,PUBLIC WORKS,100,0.0,0.000000
8,POLICE,,44,-144408906.8,3473.359617
9,PUBLIC WORKS,PUBLIC WORKS,100,0.0,0.000000


In [19]:
#MASTER SHAPE:
#Missing budget info %:


SyntaxError: invalid syntax (1136014442.py, line 1)

In [20]:
# Department Stress Score
master["dept_stress_score"] = 1 - master["percent_remaining"]

# Clean invalid values
master["dept_stress_score"] = master["dept_stress_score"].clip(lower=0, upper=1)

print(master[[
    "department",
    "percent_remaining",
    "dept_stress_score"
]].head(15))


               department  percent_remaining  dept_stress_score
0          NOT APPLICABLE        3473.359617                0.0
1        GENERAL SERVICES        3473.359617                0.0
2          NOT APPLICABLE        3473.359617                0.0
3                  POLICE        3473.359617                0.0
4   COMMUNITY DEVELOPMENT       51177.414444                0.0
5                  POLICE        3473.359617                0.0
6            PUBLIC WORKS           0.000000                1.0
7            PUBLIC WORKS           0.000000                1.0
8                  POLICE        3473.359617                0.0
9            PUBLIC WORKS           0.000000                1.0
10           PUBLIC WORKS           0.000000                1.0
11                 POLICE        3473.359617                0.0
12                   FIRE        3473.359617                0.0
13                   FIRE        3473.359617                0.0
14           PUBLIC WORKS           0.00